<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_47_telegram_bot/note_lesson_47_telegram_bot_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Урок 47 — Telegram Bot API

| Крок | Що робимо |
|---|---|
| 0 | `news_hub` уроку 47, база й двійник Telegram Bot API прямо в ноутбуці |
| 1 | update зсередини і перша команда (вправи 1–2) |
| 2 | порядок handler у роутері (вправа 3) |
| 3 | HTML: екранування і ліміт 4096 (вправи 4–5) |
| 4 | підписки й сповіщення, заблокований чат (вправи 6–7) |
| 5 | webhook: секрет у заголовку (вправи 8–9) |
| 6 | знайди помилку: ім'я в HTML (вправа 10) |

**Як працювати:** зверху вниз; перед **🔮 Прогнозом** спершу відповідай сам. Токен бота й мережа не потрібні: «Telegram» — двійник `tests/telegram_twin.py`, aiohttp-сервер у цьому ноутбуці, що відповідає у форматі Bot API. Зміни коду — у книзі курсу: [Урок 47](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_47/).

---
## 0. Проєкт, база і двійник Telegram

Клітинка завантажує `news_hub` (у Colab), ставить залежності, створює базу SQLite і запускає двійник Telegram Bot API.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_47_telegram_bot"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiogram, aiosqlite, bcrypt, fakeredis, fastapi, httpx, jwt  # noqa: F401
    from google import genai  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
os.environ.update({"DATABASE_URL": "sqlite+aiosqlite:///lesson47.db", "REDIS_URL": "fakeredis://", "LLM_PROVIDER": "fake"})
if os.path.exists("lesson47.db"):
    os.remove("lesson47.db")
result = subprocess.run([sys.executable, "-m", "alembic", "upgrade", "head"], capture_output=True, text=True)
print((result.stdout + result.stderr).strip().splitlines()[-1])

In [ ]:
import fakeredis
from aiogram.types import Update

from news_hub.bot.factory import create_bot, create_dispatcher
from news_hub.db import SessionFactory
from news_hub.llm import FakeLLM
from news_hub.models import validate_news
from news_hub.repository import NewsRepository, SubscriptionRepository
from news_hub.snapshot import load_snapshot
from tests.telegram_twin import TelegramTwin, html_error

twin = TelegramTwin()
await twin.start()
redis = fakeredis.FakeAsyncRedis(decode_responses=True)
bot = create_bot(twin.token, twin.url)                        # бот ходить на двійник, а не на api.telegram.org
dp = create_dispatcher(SessionFactory, redis, FakeLLM(), admin_ids=frozenset({1001}))


async def say(text: str, chat_id: int = 1001, **user) -> list[str]:
    """Надіслати текст боту від користувача; повертає відповіді бота цьому чату."""
    before = len(twin.sent(chat_id))
    update = Update.model_validate(twin.make_update(text, chat_id=chat_id, **user), context={"bot": bot})
    await dp.feed_update(bot, update)
    return twin.sent(chat_id)[before:]


print("двійник:", twin.url, "| бот:", (await bot.get_me()).username)

---
## 1. Update зсередини

Telegram надсилає боту **update** — JSON з повідомленням. `twin.make_update(...)` будує такий самий.

### Вправа 1

Збудуй update для тексту `/news 3` від чату `2002` з ім'ям «Анна» і збережи в `update`. Знайди в ньому: текст (`text`), id чату (`chat_id`) і тип першої «сутності» повідомлення (`entity_type`).

In [ ]:
import json

# YOUR CODE HERE

print(json.dumps(update, ensure_ascii=False, indent=1)[:500])
assert (text, chat_id, entity_type) == ("/news 3", 2002, "bot_command")
print("✅ Вправа 1 пройдена")

### Вправа 2. Перша відповідь

Збережи знімок новин у базу (`load_snapshot()` → `validate_news` → `NewsRepository.insert_new`, потім COMMIT), а тоді надішли боту `/news 3`. Збережи відповідь у `reply` і кількість нових новин у `saved`.

In [ ]:
# YOUR CODE HERE

print("збережено:", saved)
print(reply)
assert saved > 0 and reply.startswith("📰 <b>Останні новини</b> (3)") and reply.count("<a href=") == 3
print("✅ Вправа 2 пройдена")

---
## 2. Порядок handler

Шлях update крізь диспетчер: outer-middleware (ліміт) → handler по черзі → перший, чий фільтр підійшов → inner-middleware (сесія бази) → handler:

```mermaid
flowchart TD
    classDef step     fill:#263238,stroke:#90a4ae,color:#ffffff;
    classDef decision fill:#37474f,stroke:#64b5f6,color:#ffffff;
    classDef success  fill:#1b5e20,stroke:#4CAF50,color:#ffffff;
    classDef error    fill:#4e1f1f,stroke:#f44336,color:#ffffff;
    classDef warning  fill:#4a3b00,stroke:#ff9800,color:#ffffff;

    subgraph U1["«/subscribe Бюджет»"]
        direction LR
        a1{"ліміт: 1 ≤ 20?"} -- так --> a2["start, help, news,<br>digest — ні"] --> a3{"subscribe?"} -- так --> a4["сесія → INSERT<br>→ COMMIT"]
    end
    subgraph U2["«привіт»"]
        direction LR
        b1{"ліміт: 2 ≤ 20?"} -- так --> b2["жодна команда"] --> b3{"F.text?"} -- так --> b4["Не знаю такої команди"]
    end
    U1 --> U2

    class a1,a3,b1,b3 decision
    class a2,b2 step
    class a4,b4 success
```

### Вправа 3

Виведи назви handler у тому порядку, у якому їх перевіряє роутер (`dp.sub_routers[0].message.handlers`, у кожного — `.callback.__name__`), у список `order`.

**🔮 Прогноз:** що відповів би бот на `/help`, якби `unknown` (фільтр `F.text`) стояв першим?

In [ ]:
# YOUR CODE HERE

print(order)
assert order[0] == "cmd_start" and order[-1] == "unknown" and "cmd_subscribe" in order
print("✅ Вправа 3 пройдена")

<details>
<summary>Відповідь</summary>

«Не знаю такої команди» — на `/help` і на будь-яку іншу команду. aiogram бере **перший** handler, чий фільтр підійшов, а `F.text` підходить до будь-якого тексту. Тому «будь-який текст» — останнім.
</details>

---
## 3. HTML: екранування і ліміт 4096

Бот відповідає з `parse_mode=HTML`. Telegram (і двійник) відхиляє повідомлення, де «<», «>» чи «&» стоять поза тегом. `html_error(text)` з двійника каже, що не так, або повертає `None`.

### Вправа 4

Для кожного рядка з `SAMPLES` перевір `html_error(f"<b>{s}</b>")` і `html_error(f"<b>{esc(s)}</b>")`. Збережи списки `raw_ok` і `escaped_ok` (True — розмітка валідна).

In [ ]:
from news_hub.bot.formatting import esc, link, split_message

SAMPLES = ["Олена", "<Олена>", "Tom & Jerry", "ціна < 100 грн", "&amp; уже екрановано"]

# YOUR CODE HERE

for sample, raw, safe in zip(SAMPLES, raw_ok, escaped_ok):
    print(f"{sample!r:24} як є: {raw!s:5}  після esc: {safe}")
assert raw_ok == [True, False, False, False, True] and all(escaped_ok)
print("✅ Вправа 4 пройдена")

### Вправа 5. Ліміт 4096

`lines` — 300 рядків-посилань. Порівняй два способи поділу:

1. `chunks` — кожні 4000 символів (як `split_long_message` стартового `ai_bot`);
2. `parts` — `split_message(lines)` — між рядками.

Порахуй, скільки частин кожного способу **невалідні** (`bad_chunks`, `bad_parts`).

In [ ]:
lines = [link(f"https://www.rbc.ua/ukr/news/{i}.html", f"Новина {i}: бюджет & податки") for i in range(300)]
text = "\n".join(lines)

# YOUR CODE HERE

print("кожні 4000:", len(chunks), "частин, невалідних", bad_chunks)
print("між рядками:", len(parts), "частин, невалідних", bad_parts, "| довжини:", [len(p) for p in parts])
assert bad_chunks > 0 and bad_parts == 0 and all(len(p) <= 4096 for p in parts)
print("✅ Вправа 5 пройдена")

---
## 4. Підписки і сповіщення

`/subscribe <слово>` записує пару «чат + слово». Після збору `notify_subscribers(bot, subscriptions, нові_новини)` надсилає кожному чату одне повідомлення зі збігами.

### Вправа 6

Анна (`2002`) підписується на «зеленськ», Боб (`3003`) — на «погода». Збери новини вдруге так, щоб отримати **нові** — очисти базу (`DELETE FROM news` через `NewsRepository.clear`), знову `insert_new`, і виклич `notify_subscribers`. Збережи звіт у `report` і те, що отримали Анна (`anna`) і Боб (`bob`).

In [ ]:
from news_hub.notify import notify_subscribers

await say("/subscribe зеленськ", chat_id=2002, first_name="Анна")
await say("/subscribe погода", chat_id=3003, first_name="Боб")

# YOUR CODE HERE

print(report)
print(anna[:300])
print("Боб отримав останнім:", bob[:60])
assert report.chats == 1 and anna.startswith("🔔 Нові новини за підписками (зеленськ):")
assert not bob.startswith("🔔")
print("✅ Вправа 6 пройдена")

### Вправа 7. Бота заблокували

Анна заблокувала бота: `twin.blocked.add(2002)`. Повтори збір і розсилку з вправи 6. Збережи звіт у `report` і підписки Анни після розсилки у `left`.

**🔮 Прогноз:** що зробить розсилка з підписками Анни — і навіщо?

In [ ]:
twin.blocked.add(2002)

# YOUR CODE HERE

print(report, "| підписки Анни:", left)
assert report.blocked == [2002] and left == []
print("✅ Вправа 7 пройдена")

<details>
<summary>Відповідь</summary>

Telegram відповідає `403 Forbidden: bot was blocked by the user`. Розсилка видаляє підписки цього чату: інакше кожен збір знову стукав би в чат, який ніколи не відповість. Решта розсилки триває.
</details>

---
## 5. Webhook: секрет у заголовку

У webhook-режимі Telegram сам надсилає update на `POST /api/telegram/webhook` з заголовком `X-Telegram-Bot-Api-Secret-Token`. Той самий секрет застосунок передав у `setWebhook`.

### Вправа 8

Застосунок `app` з увімкненим ботом (клітинка нижче) і `httpx.AsyncClient`. Надішли `twin.make_update("/help")` на webhook:

1. без заголовка — код у `without`;
2. з правильним секретом — код у `with_secret` і відповідь бота (`answer`).

In [ ]:
import httpx

from news_hub.api import app
from news_hub.bot.settings import BotSettings

SECRET = "notebook-telegram-secret-" + "x" * 32
app.state.bot, app.state.dispatcher = bot, dp
app.state.telegram = BotSettings(token=twin.token, api_url=twin.url, webhook_url="https://news.example/hook",
                                 webhook_secret=SECRET)
api = httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://test")
HEADER = "X-Telegram-Bot-Api-Secret-Token"

# YOUR CODE HERE

print("без секрету:", without, "| з секретом:", with_secret)
print(answer[:80])
assert (without, with_secret) == (401, 200) and answer.startswith("<b>Команди</b>")
print("✅ Вправа 8 пройдена")

### Вправа 9. Адмін-команда

`/scrape snapshot` у боті збирає новини — але лише для `admin_ids` (тут `1001`). Надішли її від `4004` (`denied`) і від `1001` (`allowed`).

In [ ]:
# YOUR CODE HERE

print("4004:", denied)
print("1001:", allowed)
assert denied == "Ця команда — лише для адміністратора." and allowed.startswith("Зібрано: 168")
print("✅ Вправа 9 пройдена")

**🔮 Прогноз:** чому захист у `/scrape` — за `user_id`, а не за `chat_id`?

<details>
<summary>Відповідь</summary>

`chat_id` — розмова, `user_id` — людина. У групі один `chat_id` на всіх учасників: перевірка за чатом дала б право адміна кожному в групі, де є адмін.
</details>

---
## 6. Знайди помилку

`old_start` — handler `/start` зі стартового коду. Його тест з ім'ям «Олена» зелений.

### Вправа 10

1. Збери диспетчер `old_dp` з роутером, де лише `old_start`, і надішли `/start` від імені `Tom & Jerry` (чат `5005`). Збережи назву винятку (`error_name`), якщо `feed_update` його кине.
2. Напиши `fixed_start` — те саме привітання, але з `esc`, і перевір, що бот відповідає.

In [ ]:
from aiogram import Dispatcher, Router
from aiogram.filters import CommandStart
from aiogram.types import Message


async def old_start(message: Message) -> None:
    await message.answer(f"Привіт, <b>{message.from_user.first_name}</b>! 🤖")


def dispatcher_with(handler) -> Dispatcher:
    router = Router()
    router.message(CommandStart())(handler)
    new_dp = Dispatcher()
    new_dp.include_router(router)
    return new_dp


def update_from(name: str, chat_id: int = 5005) -> Update:
    return Update.model_validate(twin.make_update("/start", chat_id=chat_id, first_name=name), context={"bot": bot})

# YOUR CODE HERE

print("старий:", error_name, "| новий:", twin.sent(5005)[-1])
assert error_name == "TelegramBadRequest" and twin.sent(5005)[-1] == "Привіт, <b>Tom &amp; Jerry</b>! 🤖"
print("✅ Вправа 10 пройдена")

<details>
<summary>Відповідь</summary>

Ім'я вставлено в HTML без екранування: «&» поза сутністю — зламана розмітка, і Telegram відхиляє все повідомлення (`TelegramBadRequest: can't parse entities`). Бот мовчить. Тест перевіряв лише «звичайне» ім'я. Виправлення — `esc(...)` для всього, що прийшло ззовні.
</details>

In [ ]:
await api.aclose()
await bot.session.close()
await twin.stop()
app.state.bot = app.state.dispatcher = app.state.telegram = None
print("готово")

---
## Самоперевірка

1. Що таке update і хто його надсилає в режимах polling і webhook?
2. Чому `F.text` — останній handler?
3. Що станеться з повідомленням, де ім'я користувача містить «&», і як цього уникнути?
4. Навіщо ділити довгий текст між рядками, а не кожні N символів?
5. Що робить розсилка, коли чат заблокував бота, і коли Telegram відповідає 429?

## Далі

- Книга курсу: [Урок 47](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_47/) — polling і webhook, сповіщення покроково, архітектура.
- У папці `news_hub`: `pytest tests/integration/test_bot.py -v`; бот наживо на двійнику — README, розділ «Telegram-бот».
- Уроки 48–50 — Docker, Compose, CI/CD.